<a href="https://colab.research.google.com/github/Abdo15P/Fairness-Aware-KMeans/blob/main/gini-index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#NEW GINI (LEAVE THE OLD)


import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
# from preprocessing import preprocessing
# from KmeansppClass import K_Means
# from balance_measure import balance_measure
# from cluster_quality_measure import cluster_quality_measure
# from fairness_measure import fairness_measure
# from clustering_for_income import clustering_for_income

def gini_index_reassignment_combined(data, predictions, sensitive_features, centroids, k, KNN, max_swaps=1000):
    print("Starting Gini-Index Reassignment to improve fairness...")
    print(" ")

    ###########################################################################
    ################### 1- Make new Feature (race + gender) ###################
    ###########################################################################

    if len(sensitive_features.shape) == 1:
        sensitive_features = np.reshape(sensitive_features, (sensitive_features.shape[0], 1)) #reshape to column vector
    fair_preds = predictions.copy()

    combination = []
    n_sensitive_features = sensitive_features.shape[1]

    #merge sensitive attributes into one categorical combination
    for i in range(data.shape[0]):
        new_feature = ""
        for j in range(n_sensitive_features):
            new_feature = new_feature + sensitive_features[i, j]
            if j != n_sensitive_features - 1:
                new_feature = new_feature + " "
        combination.append(new_feature)


    #############################################################################
    ################### 2- get statistics for balance measure ###################
    #############################################################################

    combination = np.array(combination)
    combinations = list(set(combination))

    n_combinations = len(combinations)
    combinations_count = []

    cluster_combinations_counts = np.zeros((k, n_combinations))
    for i in range(k):
        combination_in_cluster = (combination[fair_preds == i])
        for j in range(n_combinations):
            cluster_combinations_counts[i, j] = np.sum(combination_in_cluster == combinations[j]) #count of data points belonging to combination j inside cluster i.

    #percentages of each combination within each cluster.
    cluster_combinations_perc = np.zeros_like(cluster_combinations_counts)
    for i in range(k):
        cluster_combinations_perc[i] = cluster_combinations_counts[i] / np.sum(cluster_combinations_counts[i])
    cluster_combinations_perc = cluster_combinations_perc * 100

    #golbal count of each combination
    for i in range(n_combinations):
        total_count = 0
        for j in range(k):
            total_count = total_count + cluster_combinations_counts[j, i]
        combinations_count.append(total_count)

    #golbal percentage of each combination
    combinations_count = np.array(combinations_count)
    combinations_perc = (combinations_count / np.sum(combinations_count)) * 100

    print("Initial statistics")
    print("cluster combined count", cluster_combinations_counts)
    print("cluster combined percent", cluster_combinations_perc)
    print(" ")
    print("combined count", combinations_count)
    print("combined percent", combinations_perc)
    print(" ")
    bcss, wcss = cluster_quality_measure(data, centroids, cluster_combinations_counts, fair_preds)
    initial_cluster_quality = bcss / (bcss + wcss)
    initial_fairness_quality = fairness_measure(cluster_combinations_counts, cluster_combinations_perc,
                                                combinations_perc, k)

    ##############################################################################
    ################### 3- calculate impurity ###################
    ##############################################################################

    knn = NearestNeighbors(n_neighbors=KNN, algorithm='auto', metric='euclidean')
    knn.fit(data)
    distances, indices = knn.kneighbors(data)

    idx = []
    destination_idx = []
    G = []
    for i, point in enumerate(data):
        cluster_idx = []
        gini = []
        for j in range(KNN):
            cluster_idx.append(fair_preds[indices[i, j]])

        nearest_clusters = list(set(cluster_idx))
        n_clusters_neighbours = len(nearest_clusters)

        counts = []
        cluster = []
        for unique_cluster in nearest_clusters:
            count = 0
            for f in range(len(cluster_idx)):
                if cluster_idx[f] == unique_cluster:
                    count = count + 1
            counts.append(count)
            cluster.append(unique_cluster)
            pj = count / KNN
            gini.append(pj * (1 - pj))

        sorted_candidate_clusters = sorted(zip(cluster, counts), key=lambda x:x[1], reverse = True)
        source_cluster = int(fair_preds[i])
        member_combination = combination[i]
        comb_idx = combinations.index(member_combination)
        for m, (cluster, count) in enumerate(sorted_candidate_clusters):
            if cluster != fair_preds[i]:
                candidate_dest_cluster = int(cluster)
                G.append(np.sum(gini))
                idx.append(i)
                destination_idx.append(candidate_dest_cluster)
                break
    sorted_gini = sorted(zip(idx, destination_idx, G), key=lambda x:x[2], reverse=True)

    #################################################################
    ################### 4- reassignment algorithm ###################
    #################################################################

    reassignment_count = 0
    no_change = 1
    last_cluster_quality = 0
    last_fairness_quality = 0

    #reassign until all combination deviations are below 10%.
    for i, (idx, destination_idx, _) in enumerate(sorted_gini):
        ################### \1/ measure cluster balance error from global balance ###################
        print("Candidate destinations===================== ", len(sorted_gini))
        deviations = balance_measure(cluster_combinations_perc, combinations_perc)
        if all(dev < 10 for dev in deviations):
            print("Clusters balanced")
            last_centroids = {}
            last_classes = {}
            for i in range(k):
                last_classes[i] = []
            for j, point in enumerate(data):
                for c in range(k):
                    if fair_preds[j] == c:
                        last_classes[c].append(point)
            for i in range(k):
                last_centroids[i] = np.average(last_classes[i], axis=0)

            bcss, wcss = cluster_quality_measure(data, last_centroids, cluster_combinations_counts, fair_preds)
            last_cluster_quality = bcss / (bcss + wcss)
            print("cluster_quality is ", last_cluster_quality)
            last_fairness_quality = fairness_measure(cluster_combinations_counts, cluster_combinations_perc, combinations_perc, k)
            print("fairness_quality is ", last_fairness_quality)
            print("Objective function========================================== ", last_cluster_quality + 1-last_fairness_quality)
            return fair_preds, initial_cluster_quality, last_cluster_quality, initial_fairness_quality, last_fairness_quality, no_change, 0

        no_change = 0
        print("No. of Reassignments ", reassignment_count)



        ################### \2/ reassignm cluster membership ###################
        source_cluster = int(fair_preds[idx])
        member_combination = combination[idx]  # check
        comb_idx = combinations.index(member_combination)
        if ((cluster_combinations_perc[source_cluster, comb_idx] - combinations_perc[comb_idx] < 10) or
         ((cluster_combinations_perc[source_cluster, comb_idx]) < (cluster_combinations_perc[destination_idx, comb_idx]))):

            print("Skip point")
            continue

        fair_preds[idx] = destination_idx
        reassignment_count = reassignment_count + 1
        print(reassignment_count)
        # update cluster counts and percentage
        cluster_combinations_counts[source_cluster, comb_idx] = cluster_combinations_counts[source_cluster, comb_idx] - 1
        cluster_combinations_counts[destination_idx, comb_idx] = cluster_combinations_counts[destination_idx, comb_idx] + 1

        cluster_combinations_perc = np.zeros_like(cluster_combinations_counts)
        for j in range(k):
            cluster_combinations_perc[j] = cluster_combinations_counts[j] / np.sum(cluster_combinations_counts[j])
        cluster_combinations_perc = cluster_combinations_perc * 100

        print("cluster combined count")
        for j in range(k):
            print(cluster_combinations_counts[j])
        print("cluster combined percent")
        for j in range(k):
            print(cluster_combinations_perc[j])
        print("combined count", combinations_count)
        print("combined percent", combinations_perc)
        print(" ")
        if reassignment_count >= max_swaps:
            print(f"Reached max swaps limit ({max_swaps}). Stopping.")

            # 1. Recalculate the NEW physical centroids after swapping
            last_centroids = {}
            for c in range(k):
                cluster_points = data[fair_preds == c]
                if len(cluster_points) > 0:
                    last_centroids[c] = np.average(cluster_points, axis=0)
                else:
                    last_centroids[c] = centroids[c] # Fallback if empty

            # 2. Measure quality using the NEW centroids
            final_cluster_sizes = [np.sum(fair_preds == c) for c in range(k)]
            bcss, wcss = cluster_quality_measure(data, last_centroids, final_cluster_sizes, fair_preds)

            last_cluster_quality = bcss / (bcss + wcss)
            last_fairness_quality = fairness_measure(cluster_combinations_counts, cluster_combinations_perc, combinations_perc, k)

            return fair_preds, initial_cluster_quality, last_cluster_quality, initial_fairness_quality, last_fairness_quality, 0, 0

    print(combinations)
    return fair_preds, initial_cluster_quality, last_cluster_quality, initial_fairness_quality, last_fairness_quality, no_change, 1


def main():
    data, sensitive_features, income = preprocessing()
    K = 2
    k_means = K_Means(K)
    k_means.fit(data)
    labels = np.zeros((data.shape[0],))
    for i in range(data.shape[0]):
        labels[i] = k_means.predict(data[i])

    result = pd.DataFrame({
        "cluster": labels,
        "income": income
    })

    print(pd.crosstab(result["cluster"], result["income"], normalize='index'))
    print(pd.crosstab(labels, sensitive_features[:, 0], normalize='index'))
    print(pd.crosstab(labels, sensitive_features[:, 1], normalize='index'))

    print("Initial clustering to predict income: ")
    _ = clustering_for_income(labels, income, K)

    knn = 30
    fair_predictions, initial_cluster_quality, last_cluster_quality, initial_fairness_quality, last_fairness_quality, no_change, failed = gini_index_reassignment_combined(data, labels, sensitive_features[:, 1], k_means.centroids, K, knn,10)
    for i in range(K):
        print("Initial Cluster ", i, "size ", np.sum(labels == i))
        print("Last Cluster ", i, "size ", np.sum(fair_predictions == i))
    print("Initial Clustering Quality ", initial_cluster_quality)
    print("Last Clustering Quality ", last_cluster_quality)
    print("Initial Fairness Quality ", initial_fairness_quality)
    print("Last Fairness Quality ", last_fairness_quality)
    print("No change? ", no_change)
    print("Failed? ", failed)
    print(" ")
    print("Last clustering to predict income: ")
    _ = clustering_for_income(fair_predictions, income, K)
    print("Clustering quality change = ", ((last_cluster_quality - initial_cluster_quality) / initial_cluster_quality) * 100)


if __name__ == "__main__":
    main()

Before dropping rows with missing values: 32561
After dropping rows with missing values: 30162
income      <=50K      >50K
cluster                    
0.0      0.958403  0.041597
1.0      0.591336  0.408664
col_0  Amer-Indian-Eskimo  Asian-Pac-Islander     Black     Other     White
row_0                                                                      
0.0              0.012190            0.029941  0.124105  0.011352  0.822414
1.0              0.007396            0.029467  0.069735  0.004813  0.888589
col_0    Female      Male
row_0                    
0.0    0.460917  0.539083
1.0    0.219066  0.780934
Initial clustering to predict income: 
cluster  0 percentage <=50k  95.84031692823403
cluster  0 percentage >50k  4.1596830717659605
cluster  1 percentage <=50k  59.13359943648744
cluster  1 percentage >50k  40.86640056351256
Starting Gini-Index Reassignment to improve fairness...
 
Initial statistics
cluster combined count [[ 6050.  7076.]
 [ 3732. 13304.]]
cluster combined percent